In [ ]:
import numpy
import random
import pandas
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

In [ ]:
# Define random seeds for reproducibility
SEED: int = 42
random.seed(SEED)
numpy.random.seed(SEED)

In [ ]:
# Arrythmia dataset
# https://archive.ics.uci.edu/dataset/5/arrhythmia

# Mivel 279 feature van és a 280. oszlop a célváltozó, legeneráljuk a neveket
column_names: list[str] = [f"feature_{i}" for i in range(1, 280)] + ["Class"]

# A header=None és a names=column_names mondja meg a Pandas-nak,
# hogy nincs fejléc, és használja a mi általunk generált neveket
df: pandas.DataFrame = pandas.read_csv("arrhythmia.data", header=None, names=column_names)
print(df.head())

print(f"[*] Eredeti adathalmaz mérete: {df.shape}")

# --- 1. HIÁNYZÓ ADATOK KEZELÉSE ---
print("[*] Hiányzó adatok ('?') cseréje NaN-ra...")
df.replace('?', numpy.nan, inplace=True)

# A feature_14 (az eredeti leírásban a 13-as indexű J vektor) 376 hiányzó értéket tartalmaz a 452-ből.
# Ezt a szakirodalom alapján el kell dobni, különben tönkreteszi a pótlást.
print("[*] J vektor oszlop (feature_14) eldobása a túl sok hiányzó adat miatt...")
df.drop(columns=['feature_14'], inplace=True)

# Minden oszlop numerikussá konvertálása (a '?' stringek miatt az oszlopok object típusúak lettek)
df = df.apply(pandas.to_numeric)

# --- 2. CÉLVÁLTOZÓ BINARIZÁLÁSA ---
# Eredeti: 1 = Normal, 2-16 = Különböző aritmiák
# Új célváltozó: 0 = Normal, 1 = Arrhythmia
print("[*] Célváltozó (Outcome) binarizálása: 0 (Normal) vs 1 (Arrhythmia)...")
df['Outcome'] = (df['Class'] > 1).astype(int)
df.drop(columns=['Class'], inplace=True) # Eredeti többosztályos oszlop eldobása

# --- 3. HIÁNYZÓ ÉRTÉKEK PÓTLÁSA (IMPUTATION) ---
print("[*] Maradék hiányzó értékek pótlása (Medián imputáció)...")
# Skálázást nem alkalmazunk, csak pótlást!
feature_cols = [c for c in df.columns if c != 'Outcome']

imputer = SimpleImputer(strategy='median')
df[feature_cols] = imputer.fit_transform(df[feature_cols])

# Dummy változók: Bár a 'feature_2' (Sex) kategorikus, már eleve 0/1 formátumú.
# Ha lennének többkategóriás szöveges oszlopok, itt hívnánk a pd.get_dummies(df) függvényt.

binary_like = ['feature_56', 'feature_92', 'feature_104', 'feature_139',
               'feature_195', 'feature_225', 'feature_235', 'feature_264']
for col in binary_like:
    df[col] = (df[col] != 0).astype(int)

# --- 4. TRAIN / TEST SPLIT RÉTEGZÉSSEL ---
print("[*] Tanító és teszt halmazok szétválasztása (Stratified 70/30 split)...")
SEED = 42
train_df, test_df = train_test_split(
    df,
    test_size=0.3,
    random_state=SEED,
    stratify=df["Outcome"] # Ez garantálja, hogy a 0 és 1 osztályok aránya megegyezzen a két halmazban
)

print(f"[*] Train halmaz mérete: {train_df.shape} | Arrhythmia arány: {train_df['Outcome'].mean():.2%}")
print(f"[*] Test halmaz mérete:  {test_df.shape} | Arrhythmia arány: {test_df['Outcome'].mean():.2%}")

# --- 5. MENTÉS CSV-BE ---
print("[*] CSV fájlok mentése...")
train_csv_path = "arrhythmia_preprocessed_train_data.csv"
test_csv_path  = "arrhythmia_preprocessed_test_data.csv"

train_df.to_csv(train_csv_path, index=False, float_format="%.6f")
test_df.to_csv(test_csv_path, index=False, float_format="%.6f")

print("[*] Kész! Fájlok elmentve.")